In [1]:
import pandas as pd
import re

In [2]:
df_b1 = pd.read_csv("../data/processed/bioshock_1_clean.csv", parse_dates=["review_date"])
df_b2 = pd.read_csv("../data/processed/bioshock_2_clean.csv", parse_dates=["review_date"])
df_inf = pd.read_csv("../data/processed/bioshock_infinite_clean.csv", parse_dates=["review_date"])

In [3]:
THEMES = {
    "narrative":["twist", "would you kindly", "betrayal", "obedience", "protector", "flashback", "redemption", "timeline", "multiverse", "realities", "constants", "genes", "genetic engineering", "gene altering", "autonomy", "agency", "conditioned", "kidnapping", "storytelling", "narrative", "writing", "dialogue", "story", "plot", "ending"],
    "setting" :["rapture", "underwater", "city", "art deco", "retro", "theatre", "columbia", "floating", "sky", "americana", "steampunk", "utopia", "environment", "architecture", "dystopian", "ocean", "ruins", "aesthetic", "lighthouse", "pavilion", "arcadia", "prometheus", "amusements"],
    "atmosphere" :["horror", "eerie", "scary", "tense", "immersive", "uncomfortable", "disturbing", "mystery", "dread", "claustrophobic", "twisted", "unsettling"],
    "characters" :["jack", "ryan", "atlas", "fontaine", "tenenbaum", "cohen", "little sister", "daddies", "daddy", "splicer", "delta", "eleanor", "lamb", "sofia", "sinclair", "grace", "big sister", "poole", "booker", "elizabeth", "comstock", "songbird", "twins", "daisy", "turrets", "mosquito", "zeppelin", "barrage", "handyman", "fireman", "zealot", "motorized patriot", "siren", "boy of silence", "lutece", "suchong"],
    "combat" :["plasmids", "combat", "movement", "quantum tear", "weapons", "gunplay", "shooting", "vigors", "guns", "wrench", "research camera", "drill", "rivet", "hack", "sky-hook", "adam", "eve", "tonics", "difficulty"],
    "ideology" :["capitalism", "greed", "individualism", "free will", "wealth disparity", "exploitation", "slavery", "selfishness", "political", "religious", "religion", "oppression", "collectivism", "utilitarianism", "exceptionalism", "communism", "police state", "ayn rand", "authoritarian", "right wing", "morality", "objectivism", "racism", "nationalism"],
    "visual_audio" :["lighting", "audio", "graphics", "music", "voice acting", "sound design", "style", "soundtrack"],
    "technical" :["crash", "stutter", "performance", "port", "patch", "fov", "mouse", "bugs", "optimization", "fps", "controls", "resolution", "freezing", "freeze", "glitching", "glitch", "glitches", "crashes", "crashed", "crashing", "laggy"],
    "pacing" :["repetitive", "short", "long", "boring", "tedious", "replayability", "backtrack", "drags", "filler", "padding"],
}

In [4]:
def detect_themes(text, themes=THEMES):
    text = str(text).lower()
    return {name: any(re.search(rf"\b{re.escape(kw)}\b", text) for kw in keywords)
            for name, keywords in themes.items()}

In [8]:
theme_cols = list(THEMES.keys())

In [9]:
def add_playtime_segment(frame):
    hours = frame["author.playtime_at_review"] / 60
    frame["segment"] = pd.cut(hours, bins=[0, 2, 12, float("inf")],
                              labels=["brief (<2h)", "moderate (2-12h)", "extended (12h+)"])

In [10]:
for frame in (df_b1, df_b2, df_inf):
    add_playtime_segment(frame)

In [11]:
frames = {"bioshock_1": df_b1, "bioshock_2": df_b2, "infinite": df_inf}

samples = []
for game, frame in frames.items():
    strat = frame.groupby(["voted_up", "segment"], observed=True).sample(
        n=4, random_state=42)
    strat = strat.assign(game=game)
    samples.append(strat)

validation = pd.concat(samples)[["game", "review"]].reset_index(drop=True)
len(validation)

72

In [13]:
for theme in THEMES:
    validation[theme] = ""

validation.to_csv("../data/processed/validation_sample.csv", index=False)

In [16]:
coded = pd.read_csv("../data/processed/validation_sample.csv")

detected = coded["review"].apply(detect_themes).apply(pd.Series)

for theme in theme_cols:
    coded[theme] = coded[theme].astype(int)

In [18]:
results = []
for theme in theme_cols:
    truth = coded[theme] == 1
    found = detected[theme]
    tp = (truth & found).sum()
    fp = (~truth & found).sum()
    fn = (truth & ~found).sum()
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    results.append({"theme": theme, "you_said_yes": truth.sum(),
                    "precision": round(precision, 2), "recall": round(recall, 2)})

pd.DataFrame(results)

,theme,you_said_yes,precision,recall
0,narrative,14,0.87,0.93
1,setting,8,0.80,0.50
2,atmosphere,3,1.00,0.33
3,characters,6,0.80,0.67
4,combat,14,1.00,0.57
5,ideology,2,0.50,0.50
6,visual_audio,12,0.75,0.25
7,technical,36,0.91,0.58
8,pacing,5,0.50,0.40
